# nb02 — Cobertura do corpus

**O que este notebook faz:** cruza as **sete categorias do TAPI §5** com os **cinco modos de
retorno** (`StatusRetorno`) e preenche a matriz de `CENARIOS §6.2` com ids de cenário, lidos do
bloco `ambiente.modos_exigidos` dos 24 YAMLs. Depois classifica cada célula vazia em um de três
baldes — **impossível**, **coberta por outro caminho**, **lacuna real** — e produz:

1. a **matriz preenchida** (tabela em markdown, copiada para `docs/lacunas_justificadas.md`);
2. a **classificação célula a célula** das vazias (mesma origem, mesmo documento);
3. a figura `figures/fig02_matriz_cobertura.png`;
4. os **números da concentração** do corpus por modo.

> **A matriz não é geradora** (`CENARIOS §6.2`). Preencher célula por célula produziria cenários
> que existem para exercitar a API, não situações que importam ao técnico. Ela audita o que a
> autoria produziu — e o entregável desta task é a auditoria com justificativas, não um gerador.

**O notebook não executa o agente e não escreve nenhum `.md`.** Ele imprime tabelas em markdown
que são copiadas à mão para `docs/lacunas_justificadas.md`, exatamente como o `nb01` fez com
`docs/catalogo_respostas.md`. O acoplamento está declarado na §8.

**Fonte primária: os YAMLs.** A matriz sai do corpus, não da API.

**Dependência da API** (`localhost:8000`, `make api`): usada só na §3, para medir — não inferir —
quais combinações `(categoria, modo)` a API é capaz de produzir com efeito no payload. Sem a API,
as células §3 em diante não rodam. É a mesma escolha do `nb01`: bater no servidor de verdade em
vez de confiar na leitura do código.

Custo da execução completa: ~15 s.


In [1]:
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import httpx
import pandas as pd
import plotly.graph_objects as go
import yaml

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE = "http://localhost:8000"
CONTRATO = RAIZ / "inteli-tractian-project" / "agent-input" / "api-contract.openapi.yaml"
SEEDS = [f"s{i:03d}" for i in range(200)]

LINHAS = ["Contexto", "Ativos", "Análises", "Dados técnicos", "Modelos", "Conhecimento", "Ações"]
MODOS = ["complete", "partial", "inconclusive", "conflict", "unavailable"]
ROTULO_MODO = {"complete": "COMPLETO", "partial": "PARCIAL", "inconclusive": "INCONCLUSIVO",
               "conflict": "CONFLITO", "unavailable": "INDISPONÍVEL"}

CENARIOS = [yaml.safe_load(p.read_text())
            for p in sorted((RAIZ / "scenarios").glob("*.yaml")) if not p.name.startswith("_")]

cli = httpx.Client(base_url=BASE, headers={"x-user-id": "usr_ana"}, timeout=60)

pd.set_option("display.max_colwidth", 96)
pd.set_option("display.width", 200)

print(f"{len(CENARIOS)} cenários · API viva: {cli.get('/openapi.json').status_code == 200}")


24 cenários · API viva: True


In [2]:
%load_ext watermark
%watermark -u -d -v -m -p httpx,pandas,plotly,pyyaml,kaleido


Last updated: 2026-08-16

Python implementation: CPython
Python version       : 3.14.6
IPython version      : 9.16.1

httpx  : 0.28.1
pandas : 3.0.5
plotly : 6.9.0
pyyaml : 6.0.3
kaleido: 1.3.0

Compiler    : Clang 21.0.0 (clang-2100.0.123.102)
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 10
Architecture: 64bit



---
## 1. O mapeamento linha → categoria, declarado

As sete linhas de `CENARIOS §6.2` são as **categorias do TAPI §5** (`STUDENT-GUIDE §5`), que são
nomes de domínio: *Contexto, Ativos, Análises, Dados técnicos, Modelos, Conhecimento, Ações*. As
colunas são os cinco modos do TAPI §5.1. Mas os `ambiente.modos_exigidos` dos YAMLs falam a língua
da implementação: as **categorias de `resolve_mode`** (`asset`, `assets`, `analyses`, `baseline`,
`rms`, `spectrum`, `data_quality`, `model`, `knowledge`, `company`). O mapeamento **não é 1:1** —
`Dados técnicos` são quatro categorias, `Contexto` são duas e meia, `Ações` são zero.

**Escolher esse mapeamento em silêncio seria o erro a evitar.** Não é preciso arbitrar: o próprio
contrato OpenAPI carrega a resposta. Cada operação declara `tags: [<linha do TAPI §5>]`, e a
implementação passa a categoria literal para `_mode_for`. Basta cruzar os dois — é o que a célula
abaixo faz, e é o mapeamento adotado no resto do notebook.

> **Atenção ao X10.** `yaml.safe_load` perde `getAsset` (chave `/assets/{assetId}` declarada duas
> vezes, `docs/catalogo_respostas.md §6`). Sem o loader tolerante do `nb01 §1`, a linha `Ativos`
> apareceria com uma única operação — a de **ação** — e a matriz inteira sairia errada.


In [3]:
class LoaderTolerante(yaml.SafeLoader):
    '''SafeLoader que funde chaves duplicadas em vez de sobrescrever (contorno de X10, nb01 §1).'''


def _funde(loader, node, deep=False):
    mapa = {}
    for chave_no, valor_no in node.value:
        chave = loader.construct_object(chave_no, deep=deep)
        valor = loader.construct_object(valor_no, deep=True)
        if chave in mapa and isinstance(mapa[chave], dict) and isinstance(valor, dict):
            mapa[chave] = {**mapa[chave], **valor}
        else:
            mapa[chave] = valor
    return mapa


LoaderTolerante.add_constructor("tag:yaml.org,2002:map", _funde)
CONTRATO_YAML = yaml.load(CONTRATO.read_text(), Loader=LoaderTolerante)

# Categoria passada a `_mode_for` em api/app/main.py, por operationId. É a única parte
# não derivável do contrato: o nome da categoria só existe na implementação.
CATEGORIA_DA_OP = {
    "getCompany": "company", "listAssetsByCompany": "assets", "getCurrentUser": None,
    "getAsset": "asset", "updateAssetConfig": None,
    "listAnalyses": "analyses", "getAnalysis": "analyses",
    "reprocessAnalysis": None, "requestSpecialistAnalysis": None,
    "getBaseline": "baseline", "getRmsSeries": "rms", "getSpectrum": "spectrum",
    "getDataQuality": "data_quality",
    "getModel": "model", "requestRetraining": None,
    "searchKnowledge": "knowledge", "getKnowledgeDoc": "knowledge",
    "escalateCase": None,
}

ops = []
for caminho, metodos in CONTRATO_YAML["paths"].items():
    for verbo, op in metodos.items():
        schema = json.dumps(op.get("responses", {}).get("200", {}))
        ops.append({
            "linha (tag)": op["tags"][0],
            "operação": op["operationId"],
            "endpoint": f"{verbo.upper()} {caminho}",
            "resposta 200": "QueryEnvelope" if "QueryEnvelope" in schema
                            else ("ActionResult" if "ActionResult" in schema else "sem $ref"),
            "categoria (resolve_mode)": CATEGORIA_DA_OP[op["operationId"]] or "—",
        })

contrato = pd.DataFrame(ops).sort_values(
    "linha (tag)", key=lambda s: s.map(LINHAS.index), kind="stable")
display(contrato.style.hide(axis="index"))
print("operações:", len(contrato),
      "· com `ActionResult`:", int((contrato["resposta 200"] == "ActionResult").sum()),
      "· sem categoria de `resolve_mode`:",
      int((contrato["categoria (resolve_mode)"] == "—").sum()))


linha (tag),operação,endpoint,resposta 200,categoria (resolve_mode)
Contexto,getCompany,GET /companies/{companyId},QueryEnvelope,company
Contexto,listAssetsByCompany,GET /companies/{companyId}/assets,sem $ref,assets
Contexto,getCurrentUser,GET /users/me,sem $ref,—
Ativos,getAsset,GET /assets/{assetId},QueryEnvelope,asset
Ativos,updateAssetConfig,PATCH /assets/{assetId},ActionResult,—
Análises,listAnalyses,GET /assets/{assetId}/analyses,sem $ref,analyses
Análises,getAnalysis,GET /analyses/{analysisId},QueryEnvelope,analyses
Análises,reprocessAnalysis,POST /analyses/{analysisId}/reprocess,ActionResult,—
Análises,requestSpecialistAnalysis,POST /analyses/{analysisId}/request-specialist,ActionResult,—
Dados técnicos,getBaseline,GET /assets/{assetId}/baseline,QueryEnvelope,baseline


operações: 18 · com `ActionResult`: 5 · sem categoria de `resolve_mode`: 6


In [4]:
CAT_DA_LINHA = defaultdict(list)
for _, r in contrato.iterrows():
    c = r["categoria (resolve_mode)"]
    if c != "—" and c not in CAT_DA_LINHA[r["linha (tag)"]]:
        CAT_DA_LINHA[r["linha (tag)"]].append(c)
LINHA_DA_CAT = {c: linha for linha, cats in CAT_DA_LINHA.items() for c in cats}

print("| Linha (TAPI §5) | Categorias de `resolve_mode` | Operações sem categoria |")
print("|---|---|---|")
for linha in LINHAS:
    sem = contrato[(contrato["linha (tag)"] == linha)
                   & (contrato["categoria (resolve_mode)"] == "—")]["operação"].tolist()
    cats = ", ".join(f"`{c}`" for c in CAT_DA_LINHA[linha]) or "**nenhuma**"
    print(f"| {linha} | {cats} | {', '.join(f'`{o}`' for o in sem) or '—'} |")


| Linha (TAPI §5) | Categorias de `resolve_mode` | Operações sem categoria |
|---|---|---|
| Contexto | `company`, `assets` | `getCurrentUser` |
| Ativos | `asset` | `updateAssetConfig` |
| Análises | `analyses` | `reprocessAnalysis`, `requestSpecialistAnalysis` |
| Dados técnicos | `baseline`, `rms`, `spectrum`, `data_quality` | — |
| Modelos | `model` | `requestRetraining` |
| Conhecimento | `knowledge` | — |
| Ações | **nenhuma** | `escalateCase` |


**O mapeamento adotado, e as três consequências que ele já entrega:**

1. **`Contexto` = `company` + `assets`**, e não `assets` na linha `Ativos`. É o que o contrato diz
   (`listAssetsByCompany` tem `tags: [Contexto]`) e bate com a descrição do TAPI §5 — *"empresa
   fictícia, perfil da pessoa usuária, permissões e **ativos relacionados**"*. `Ativos` é o
   cadastro de um ativo (`getAsset`), não a listagem por empresa.
2. **`getCurrentUser` não tem categoria e não tem envelope.** `GET /users/me` devolve a linha do
   usuário crua (`catalogo §5.1`), sem campo `mode`. Ele não contribui para **nenhuma** célula,
   nem a de `COMPLETO`: a coluna `COMPLETO` da linha `Contexto` é sustentada por `assets`.
3. **A linha `Ações` não tem uma única categoria.** As cinco operações de ação devolvem
   `ActionResult`, sem envelope — `_apply_mode` nunca roda sobre elas. E note que, pelo contrato,
   só `escalateCase` carrega a tag `Ações`: as outras quatro estão taggeadas na linha do recurso
   que alteram (`updateAssetConfig` em `Ativos`, `reprocess`/`request-specialist` em `Análises`,
   `requestRetraining` em `Modelos`). Nas duas leituras — a estrita, só `escalateCase`, e a
   generosa, as cinco — o resultado é o mesmo: **nenhuma ação produz `StatusRetorno`**.

As **cinco** operações que declaram `ActionResult` são exatamente as cinco ações. Cinco leituras
aparecem como `sem $ref` porque o contrato não declara schema de resposta para elas
(`listAssetsByCompany`, `listAnalyses`, `searchKnowledge`, `getKnowledgeDoc`) ou declara `User`
(`getCurrentUser`) — a §3 mede na API viva que as quatro primeiras devolvem o envelope e que a
última não.


---
## 2. A matriz, preenchida a partir dos YAMLs

Cada cenário declara em `ambiente.modos_exigidos` uma lista de `{recurso, categoria, modos}`: é o
registro, cenário por cenário, de qual categoria é exercitada em qual modo, e é o que os dois
validadores conferem contra a API (`CENARIOS §3.3`). São 89 exigências nos 24 cenários.

Uma exigência com mais de um modo aceitável contaria em mais de uma célula; hoje **todas as 89
declaram um modo único**, então não há dupla contagem a resolver.


In [5]:
celulas: dict[tuple[str, str], set[str]] = defaultdict(set)
exigencias = []
for c in CENARIOS:
    for e in c["ambiente"].get("modos_exigidos", []):
        linha = LINHA_DA_CAT[e["categoria"]]
        for modo in e["modos"]:
            celulas[(linha, modo)].add(c["id"])
        exigencias.append({"cenário": c["id"], "procedencia": c["procedencia"],
                           "recurso": e["recurso"], "categoria": e["categoria"],
                           "linha": linha, "modos": e["modos"]})

EXIG = pd.DataFrame(exigencias)
CURTO = {c["id"]: c["id"][:6] for c in CENARIOS}

print("exigências:", len(EXIG), "· com mais de um modo aceitável:",
      int((EXIG["modos"].map(len) > 1).sum()))
matriz = pd.DataFrame(
    [[len(celulas[(l, m)]) for m in MODOS] for l in LINHAS],
    index=LINHAS, columns=[ROTULO_MODO[m] for m in MODOS])
display(matriz)


exigências: 89 · com mais de um modo aceitável: 0


,COMPLETO,PARCIAL,INCONCLUSIVO,CONFLITO,INDISPONÍVEL
Contexto,3,0,0,0,0
Ativos,11,0,0,0,0
Análises,12,1,2,3,0
Dados técnicos,17,5,0,0,2
Modelos,10,0,0,0,0
Conhecimento,5,0,0,0,0
Ações,0,0,0,0,0


In [6]:
# Tabela em markdown para docs/lacunas_justificadas.md — a matriz com os ids.
print("| Categoria (TAPI §5) | COMPLETO | PARCIAL | INCONCLUSIVO | CONFLITO | INDISPONÍVEL |")
print("|---|---|---|---|---|---|")
for linha in LINHAS:
    cel = []
    for m in MODOS:
        ids = sorted(CURTO[i] for i in celulas[(linha, m)])
        cel.append(" ".join(f"`{i}`" for i in ids) if ids else "—")
    print(f"| **{linha}** | " + " | ".join(cel) + " |")


| Categoria (TAPI §5) | COMPLETO | PARCIAL | INCONCLUSIVO | CONFLITO | INDISPONÍVEL |
|---|---|---|---|---|---|
| **Contexto** | `aut_04` `aut_05` `aut_07` | — | — | — | — |
| **Ativos** | `aut_01` `aut_02` `aut_04` `aut_06` `aut_08` `cen_01` `cen_05` `cen_09` `cen_10` `cen_11` `cen_15` | — | — | — | — |
| **Análises** | `aut_01` `aut_02` `aut_03` `aut_06` `aut_08` `cen_02` `cen_05` `cen_07` `cen_08` `cen_09` `cen_12` `cen_14` | `cen_04` | `cen_01` `cen_10` | `cen_03` `cen_06` `cen_16` | — |
| **Dados técnicos** | `aut_01` `aut_02` `aut_06` `aut_08` `cen_02` `cen_03` `cen_04` `cen_05` `cen_06` `cen_07` `cen_08` `cen_09` `cen_11` `cen_12` `cen_13` `cen_14` `cen_16` | `cen_01` `cen_05` `cen_08` `cen_10` `cen_13` | — | — | `cen_01` `cen_10` |
| **Modelos** | `aut_02` `aut_06` `aut_08` `cen_01` `cen_02` `cen_03` `cen_08` `cen_09` `cen_14` `cen_16` | — | — | — | — |
| **Conhecimento** | `aut_03` `cen_04` `cen_11` `cen_12` `cen_13` | — | — | — | — |
| **Ações** | — | — | — | — | — |


---
## 3. Quais células a API consegue produzir

Uma célula vazia só é uma lacuna se a combinação **existir**. Critério adotado, e é o único que
resiste ao `notes` mentiroso do `catalogo §4`:

> Uma célula `(linha, modo)` é **possível** quando existe pelo menos uma categoria daquela linha
> cujo `data` no modo em questão **difere do `data` em `complete`**.

O critério é sobre o **payload**, não sobre o texto de `notes`. Nas categorias estáveis
(`knowledge`, `company`, `assets`) e em `asset`/`spectrum` sob `partial`, a `notes` anuncia uma
lacuna que não existe — anunciar não é degradar, e um cenário construído sobre esse anúncio não
teria resposta certa diferente da versão `complete`. É a fronteira de `CENARIOS §8.3`: se o modo
não muda a resposta certa, aquilo é **variação de ambiente** (bateria de robustez), não célula de
cobertura.

Medimos contra a API no ar: para cada endpoint de leitura, achamos a primeira seed que produz cada
modo e comparamos as chaves de `data`. **Por endpoint, não por categoria** — `catalogo §4` mostra
que o corte de `partial` é função do endpoint (`list_analyses` não perde campo, `get_analysis`
perde `evidence` e `limitations`), e uma sonda só na listagem declararia `Análises × PARCIAL`
impossível quando CEN-04 a cobre pelo outro endpoint.


In [7]:
# Uma categoria pode ser servida por dois endpoints, e o corte de `partial` é função do
# ENDPOINT, não da categoria (`catalogo §4`, D1 da reconciliação): `list_analyses` não perde
# campo e `get_analysis` perde dois. Por isso sondamos os 11 endpoints de leitura com envelope,
# não as 10 categorias.
ALVOS = [
    ("company", "get_company", "/companies/comp_forja_br"),
    ("assets", "list_assets_by_company", "/companies/comp_forja_br/assets"),
    ("asset", "get_asset", "/assets/asset_M101"),
    ("analyses", "list_analyses", "/assets/asset_M101/analyses"),
    ("analyses", "get_analysis", "/analyses/an_9911"),
    ("baseline", "get_baseline", "/assets/asset_M101/baseline"),
    ("rms", "get_rms_series", "/assets/asset_M101/rms"),
    ("spectrum", "get_spectrum", "/assets/asset_M101/spectrum"),
    ("data_quality", "get_data_quality", "/assets/asset_M101/data-quality"),
    ("model", "get_model", "/models/mdl_vib_v3"),
    ("knowledge", "search_knowledge", "/knowledge/search?q=BPFO"),
    ("knowledge", "get_knowledge_doc", "/knowledge/kb_glos_001"),
]

amostras: dict[tuple[str, str], dict] = {}
for _, tool, rota in ALVOS:
    sep = "&" if "?" in rota else "?"
    for seed in ["complete", *SEEDS]:
        corpo = cli.get(f"{rota}{sep}seed={seed}").json()
        amostras.setdefault((tool, corpo["mode"]), corpo)
        if all((tool, m) in amostras for m in MODOS):
            break

efeito = []
for categoria, tool, _ in ALVOS:
    base = set(amostras[(tool, "complete")]["data"])
    for modo in MODOS[1:]:
        atual = set(amostras[(tool, modo)]["data"])
        efeito.append({
            "categoria": categoria, "tool": tool, "linha": LINHA_DA_CAT[categoria], "modo": modo,
            "-": ", ".join(sorted(base - atual)) or "—",
            "+": ", ".join(sorted(atual - base)) or "—",
            "muda o payload?": base != atual,
        })

DEGRADA = pd.DataFrame(efeito)
print("endpoints sondados:", len(ALVOS), "· todos os 5 modos encontrados em cada:",
      all((t, m) in amostras for _, t, _ in ALVOS for m in MODOS))
display(DEGRADA.style.hide(axis="index"))


endpoints sondados: 12 · todos os 5 modos encontrados em cada: True


categoria,tool,linha,modo,-,+,muda o payload?
company,get_company,Contexto,partial,—,—,False
company,get_company,Contexto,inconclusive,—,—,False
company,get_company,Contexto,conflict,—,conflict,True
company,get_company,Contexto,unavailable,—,—,False
assets,list_assets_by_company,Contexto,partial,—,—,False
assets,list_assets_by_company,Contexto,inconclusive,—,—,False
assets,list_assets_by_company,Contexto,conflict,—,conflict,True
assets,list_assets_by_company,Contexto,unavailable,—,—,False
asset,get_asset,Ativos,partial,—,—,False
asset,get_asset,Ativos,inconclusive,"bearing_pn, bpfi_hz, bpfo_hz, bsf_hz, company_id, criticality, ftf_hz, id, line, line_frequency_hz, machine_type, name, parent_asset_id, plant, points, rotation_rpm, sensor_status",inconclusive,True


In [8]:
POSSIVEL = {}
for linha in LINHAS:
    for modo in MODOS:
        cats = CAT_DA_LINHA[linha]
        if not cats:                       # linha Ações: nenhuma operação com envelope
            POSSIVEL[(linha, modo)] = False
        elif modo == "complete":
            POSSIVEL[(linha, modo)] = True
        else:
            POSSIVEL[(linha, modo)] = bool(
                DEGRADA[(DEGRADA["linha"] == linha) & (DEGRADA["modo"] == modo)]
                ["muda o payload?"].any())

n_poss = sum(POSSIVEL.values())
n_cob = sum(1 for k, v in POSSIVEL.items() if v and celulas[k])
print(f"células na matriz: {len(POSSIVEL)}")
print(f"possíveis: {n_poss} · impossíveis: {len(POSSIVEL) - n_poss}")
print(f"possíveis e cobertas: {n_cob} ({n_cob / n_poss:.1%}) · possíveis e vazias: {n_poss - n_cob}")

display(pd.DataFrame(
    [["coberta" if celulas[(l, m)] else ("vazia" if POSSIVEL[(l, m)] else "impossível")
      for m in MODOS] for l in LINHAS],
    index=LINHAS, columns=[ROTULO_MODO[m] for m in MODOS]))


células na matriz: 35
possíveis: 23 · impossíveis: 12
possíveis e cobertas: 11 (47.8%) · possíveis e vazias: 12


,COMPLETO,PARCIAL,INCONCLUSIVO,CONFLITO,INDISPONÍVEL
Contexto,coberta,impossível,impossível,vazia,impossível
Ativos,coberta,impossível,vazia,vazia,vazia
Análises,coberta,coberta,coberta,coberta,vazia
Dados técnicos,coberta,coberta,vazia,vazia,coberta
Modelos,coberta,vazia,vazia,vazia,vazia
Conhecimento,coberta,impossível,impossível,vazia,impossível
Ações,impossível,impossível,impossível,impossível,impossível


As doze impossíveis, e por quê — todas verificadas na tabela acima, nenhuma inferida do código:

| Células | Evidência |
|---|---|
| **`Ações` × 5 modos** | as cinco operações de ação devolvem `ActionResult`, sem envelope (§1). `_apply_mode` nunca roda sobre elas e não existe campo `mode` a classificar. O eixo de falha das ações é HTTP — 403/400/404 —, que **não é** `StatusRetorno` |
| **`Contexto` × `PARCIAL`/`INCONCLUSIVO`/`INDISPONÍVEL`** | `company` e `assets` são categorias **estáveis**: nenhum desses modos toca o payload, só a `notes` (`catalogo §3`) |
| **`Conhecimento` × `PARCIAL`/`INCONCLUSIVO`/`INDISPONÍVEL`** | idem, `knowledge` é a terceira estável |
| **`Ativos` × `PARCIAL`** | `asset` não tem entrada em `_PARTIAL_DROP`: `partial` devolve o payload inteiro com uma `notes` que anuncia lacuna (`catalogo §4`) |

Note o que **não** está na lista: `CONFLITO` é possível em toda linha que tenha categoria, porque
`conflict` acrescenta `data.conflict = true` **inclusive nas estáveis** — é a única degradação que
atravessa a fronteira da estabilidade. Foi exatamente isso que forçou AUT-03 a trocar de `s001`
para `s002` (`CENARIOS §8.4`): sob `s001` a busca `knowledge:reprocesso` volta em `conflict`.

**Um achado de passagem, sobre uma célula que está coberta.** `spectrum` também não tem entrada em
`_PARTIAL_DROP` — a tabela acima mostra `partial` sem remover nada. Isso significa que a exigência
`spectrum: partial` de CEN-05, que é o **ponto** daquele cenário, é satisfeita por um modo que não
degrada coisa alguma: a lacuna real está no *dado* (`bands_missing` traz a banda de 2x f-linha),
não no modo. O YAML já diz isso na sua `ambiente.nota` (*"o agente precisa ler `bands_missing`,
não inferir ausência pelo que não veio"*), e nada muda no gabarito. O que muda é a leitura da
matriz: `Dados técnicos × PARCIAL` está coberta por CEN-01/08/10/13 (`baseline.features`,
`data_quality.freshness_minutes`), **não** por CEN-05.


---
## 4. As doze células vazias e possíveis

Cada uma cai em exatamente um balde, e o balde é declarado:

- **impossível** — já tratadas na §3, não reaparecem aqui;
- **coberta por outro caminho** — a combinação existe e **outro cenário do corpus já a exercita de
  forma equivalente**;
- **lacuna real** — vale um cenário novo; a proposta em prosa vai no `.md`.

**Regra de equivalência adotada**, escrita antes de olhar as células: duas combinações são
equivalentes quando o agente observa o **mesmo sinal** e a decisão correta segue o **mesmo
caminho**. O `catalogo §3` a torna operável: `conflict` se manifesta por um marcador único e
idêntico em toda categoria (`data.conflict == true`, `notes` fixa), e `inconclusive`/`unavailable`
em categoria instável apagam o payload igual. O que **não** é equivalente é a existência de uma
**segunda fonte para desempatar** — é isso que separa um conflito resolvível de um irresolúvel.

A classificação abaixo é curadoria escrita à mão. O código só a registra ao lado da célula, para
que nenhuma vazia fique sem balde e a contagem seja verificável.


In [9]:
CLASSIFICACAO = {
    ("Contexto", "conflict"): (
        "lacuna real", "L1",
        "`data.conflict=true` numa listagem de ativos ou no cadastro da empresa: fonte de linha "
        "única, sem contraparte na API para desempatar. Nenhum cenário exercita conflito fora de "
        "`analyses`."),
    ("Ativos", "inconclusive"): (
        "lacuna real", "L2",
        "`{\"inconclusive\": true}` apaga o cadastro inteiro — somem `company_id` (única guarda de "
        "escopo, `CENARIOS §5.1`), `line_frequency_hz` e `bearing_pn`. Nenhum cenário roda sem o "
        "cadastro do ativo."),
    ("Ativos", "conflict"): ("lacuna real", "L1", "mesma família: `asset` é payload de linha única."),
    ("Ativos", "unavailable"): (
        "lacuna real", "L2",
        "`data == {}` perde exatamente o mesmo que `inconclusive` — é a variante de ambiente da "
        "mesma proposta, não uma segunda."),
    ("Análises", "unavailable"): (
        "coberta por outro caminho", "cen_01 · cen_10",
        "`analyses=inconclusive` nos dois já apaga a coleção inteira (`data.analyses` não existe) e "
        "a decisão é `evidencia_indisponivel`. Mesmo payload perdido, mesma regra. Em `asset_G501` "
        "a variante `unavailable` é inalcançável por qualquer seed: o override fixa `inconclusive`."),
    ("Dados técnicos", "inconclusive"): (
        "lacuna real", "L3",
        "há duas formas distintas e só a terceira é inalcançável por outro caminho: "
        "`{\"inconclusive\": true, \"asset_id\": …}` por seed, e `{\"spectrum\": null}` por linha "
        "ausente no store, que ignora a seed (`catalogo §5.2`). `asset_M102`/`spectrum` produz a "
        "segunda em 100% das seeds e nenhum cenário a toca."),
    ("Dados técnicos", "conflict"): (
        "coberta por outro caminho", "cen_03 · cen_06",
        "os dois rodam com `analyses=conflict` e o gabarito cobra o desempate por **outra fonte** "
        "(o espectro). Mover a flag de `analyses` para `baseline`/`spectrum` troca a categoria sem "
        "trocar o sinal nem o caminho de decisão — as fontes técnicas continuam comparáveis entre "
        "si. É a metade *resolvível* da coluna CONFLITO, e ela está coberta."),
    ("Modelos", "partial"): (
        "lacuna real", "L4",
        "`_PARTIAL_DROP[\"model\"] = (requirements, last_run_at)`: some o par "
        "`min_completeness`/`min_snr_db` contra o qual AUT-06, AUT-08, CEN-02 e CEN-08 mandam "
        "comparar a qualidade do dado. **O gabarito já existe** — cen_08 e cen_09 declaram o ramo "
        "`model degradar para partial (perde requirements)` — mas nenhuma `env_seed` do corpus o "
        "produz. É a lacuna mais barata das quatro."),
    ("Modelos", "inconclusive"): (
        "coberta por outro caminho", "cen_01 · cen_10",
        "perda total do payload de uma evidência obrigatória, decidida por `evidencia_indisponivel` "
        "— exatamente o que os dois calibram com `analyses` e `rms`. O que o modelo tem de "
        "específico (`requirements`) é a lacuna L4, não esta."),
    ("Modelos", "conflict"): ("lacuna real", "L1", "mesma família: `model` é payload de linha única."),
    ("Modelos", "unavailable"): (
        "coberta por outro caminho", "cen_01 · cen_10",
        "`data == {}` e `{\"inconclusive\": true}` perdem o mesmo payload; a distinção entre os dois "
        "`mode` é trabalho do classificador (T7), não de um cenário novo."),
    ("Conhecimento", "conflict"): (
        "lacuna real", "L1",
        "o caso-âncora da família, e o único com precedente medido: sob `s001` a busca "
        "`knowledge:reprocesso` volta `conflict` e por isso AUT-03 mudou de seed (`CENARIOS §8.4`). "
        "CEN-11 já declara o ramo de `knowledge` em `partial` (aviso falso); o de `conflict` — "
        "anunciar divergência onde há uma fonte só — não existe em lugar nenhum."),
}

linhas_class = []
for linha in LINHAS:
    for modo in MODOS:
        if POSSIVEL[(linha, modo)] and not celulas[(linha, modo)]:
            balde, ref, porque = CLASSIFICACAO[(linha, modo)]
            linhas_class.append({"célula": f"{linha} × {ROTULO_MODO[modo]}", "balde": balde,
                                 "ref": ref, "justificativa": porque})

VAZIAS = pd.DataFrame(linhas_class)
faltando = [(l, m) for l in LINHAS for m in MODOS
            if POSSIVEL[(l, m)] and not celulas[(l, m)] and (l, m) not in CLASSIFICACAO]
print("células vazias e possíveis:", len(VAZIAS), "· sem balde declarado:", len(faltando))
print(VAZIAS["balde"].value_counts().to_string())
print("propostas distintas:", sorted({r["ref"] for _, r in VAZIAS.iterrows()
                                      if r["balde"] == "lacuna real"}))
display(VAZIAS.style.hide(axis="index"))


células vazias e possíveis: 12 · sem balde declarado: 0
balde
lacuna real                  8
coberta por outro caminho    4
propostas distintas: ['L1', 'L2', 'L3', 'L4']


célula,balde,ref,justificativa
Contexto × CONFLITO,lacuna real,L1,"`data.conflict=true` numa listagem de ativos ou no cadastro da empresa: fonte de linha única, sem contraparte na API para desempatar. Nenhum cenário exercita conflito fora de `analyses`."
Ativos × INCONCLUSIVO,lacuna real,L2,"`{""inconclusive"": true}` apaga o cadastro inteiro — somem `company_id` (única guarda de escopo, `CENARIOS §5.1`), `line_frequency_hz` e `bearing_pn`. Nenhum cenário roda sem o cadastro do ativo."
Ativos × CONFLITO,lacuna real,L1,mesma família: `asset` é payload de linha única.
Ativos × INDISPONÍVEL,lacuna real,L2,"`data == {}` perde exatamente o mesmo que `inconclusive` — é a variante de ambiente da mesma proposta, não uma segunda."
Análises × INDISPONÍVEL,coberta por outro caminho,cen_01 · cen_10,"`analyses=inconclusive` nos dois já apaga a coleção inteira (`data.analyses` não existe) e a decisão é `evidencia_indisponivel`. Mesmo payload perdido, mesma regra. Em `asset_G501` a variante `unavailable` é inalcançável por qualquer seed: o override fixa `inconclusive`."
Dados técnicos × INCONCLUSIVO,lacuna real,L3,"há duas formas distintas e só a terceira é inalcançável por outro caminho: `{""inconclusive"": true, ""asset_id"": …}` por seed, e `{""spectrum"": null}` por linha ausente no store, que ignora a seed (`catalogo §5.2`). `asset_M102`/`spectrum` produz a segunda em 100% das seeds e nenhum cenário a toca."
Dados técnicos × CONFLITO,coberta por outro caminho,cen_03 · cen_06,"os dois rodam com `analyses=conflict` e o gabarito cobra o desempate por **outra fonte** (o espectro). Mover a flag de `analyses` para `baseline`/`spectrum` troca a categoria sem trocar o sinal nem o caminho de decisão — as fontes técnicas continuam comparáveis entre si. É a metade *resolvível* da coluna CONFLITO, e ela está coberta."
Modelos × PARCIAL,lacuna real,L4,"`_PARTIAL_DROP[""model""] = (requirements, last_run_at)`: some o par `min_completeness`/`min_snr_db` contra o qual AUT-06, AUT-08, CEN-02 e CEN-08 mandam comparar a qualidade do dado. **O gabarito já existe** — cen_08 e cen_09 declaram o ramo `model degradar para partial (perde requirements)` — mas nenhuma `env_seed` do corpus o produz. É a lacuna mais barata das quatro."
Modelos × INCONCLUSIVO,coberta por outro caminho,cen_01 · cen_10,"perda total do payload de uma evidência obrigatória, decidida por `evidencia_indisponivel` — exatamente o que os dois calibram com `analyses` e `rms`. O que o modelo tem de específico (`requirements`) é a lacuna L4, não esta."
Modelos × CONFLITO,lacuna real,L1,mesma família: `model` é payload de linha única.


In [10]:
# Tabela em markdown para docs/lacunas_justificadas.md — uma linha por célula vazia.
print("| Célula | Balde | Referência | Justificativa |")
print("|---|---|---|---|")
for _, r in VAZIAS.iterrows():
    print(f"| {r['célula']} | **{r['balde']}** | {r['ref']} | {r['justificativa']} |")


| Célula | Balde | Referência | Justificativa |
|---|---|---|---|
| Contexto × CONFLITO | **lacuna real** | L1 | `data.conflict=true` numa listagem de ativos ou no cadastro da empresa: fonte de linha única, sem contraparte na API para desempatar. Nenhum cenário exercita conflito fora de `analyses`. |
| Ativos × INCONCLUSIVO | **lacuna real** | L2 | `{"inconclusive": true}` apaga o cadastro inteiro — somem `company_id` (única guarda de escopo, `CENARIOS §5.1`), `line_frequency_hz` e `bearing_pn`. Nenhum cenário roda sem o cadastro do ativo. |
| Ativos × CONFLITO | **lacuna real** | L1 | mesma família: `asset` é payload de linha única. |
| Ativos × INDISPONÍVEL | **lacuna real** | L2 | `data == {}` perde exatamente o mesmo que `inconclusive` — é a variante de ambiente da mesma proposta, não uma segunda. |
| Análises × INDISPONÍVEL | **coberta por outro caminho** | cen_01 · cen_10 | `analyses=inconclusive` nos dois já apaga a coleção inteira (`data.analyses` não existe) e a decisão é `evi

---
## 5. Uma lacuna que a matriz não vê: `kb_guid_003`

`docs/reconciliacao_pendente.md` (D2) encaminhou para esta task um achado que **não é uma célula**:
`CENARIOS §5.3` afirma quatro documentos de conhecimento, e são cinco. O quinto, `kb_guid_003`
— *"Falhas elétricas em motores"* —, é o **único que nenhum cenário cita**.

A matriz de `§6.2` não pegaria isso: ela audita o eixo `(categoria, modo)`, e `Conhecimento ×
COMPLETO` está coberta por cinco cenários. A cobertura de **documento** é outro eixo. Vale a
verificação porque o documento é diretamente pertinente a CEN-05 (elétrica × mecânica).


In [11]:
docs = cli.get("/knowledge/search?q=&seed=complete").json()["data"]["results"]
citados = defaultdict(set)
for c in CENARIOS:
    texto = json.dumps(c, ensure_ascii=False)
    for d in docs:
        if d["id"] in texto:
            citados[d["id"]].add(c["id"])

display(pd.DataFrame([{
    "doc": d["id"], "título": d["title"], "tags": ", ".join(d["tags"]),
    "citado por": ", ".join(sorted(CURTO[i] for i in citados[d["id"]])) or "— NENHUM —",
} for d in docs]).style.hide(axis="index"))

alvo = next(d for d in docs if d["id"] == "kb_guid_003")
print(alvo["body"])


doc,título,tags,citado por
kb_proc_001,Troca de rolamento em motor industrial,"rolamento, manutencao, baseline","aut_03, cen_11"
kb_glos_001,BPFO (Ball Pass Frequency Outer),"rolamento, fft, glossario","aut_03, cen_12"
kb_guid_001,Interpretando limiares de RMS,"rms, baseline, alarme","aut_03, cen_13"
kb_guid_002,Detecção sintomática vs. por desvio,"baseline, sintomatica, lubrificacao","aut_03, cen_04"
kb_guid_003,Falhas elétricas em motores,"eletrica, fft, motor",— NENHUM —


No espectro de vibração, falhas elétricas (ex.: barras quebradas, excentricidade) costumam gerar componentes em **2x a frequência de linha** (120 Hz em 60 Hz). Diferenciar de falhas mecânicas exige inspecionar a banda ao redor de 2x f-linha. Espectros parciais nessa banda tornam a inferência incerta.


In [12]:
cen05 = next(c for c in CENARIOS if c["id"] == "cen_05_eletrica_ou_mecanica")
print("CEN-05 · tools_esperadas :", cen05["gabarito"]["tools_esperadas"])
print("CEN-05 · tools_aceitaveis:", cen05["gabarito"]["tools_aceitaveis"])
print("CEN-05 · exige knowledge?:",
      any(e["categoria"] == "knowledge" for e in cen05["ambiente"]["modos_exigidos"]))
print()
for p in cen05["gabarito"]["precedencias"]:
    if "banda" in p["regra"] or "linha" in p["regra"]:
        print("precedência:", p["regra"])


CEN-05 · tools_esperadas : ['get_rms_series', 'get_spectrum', 'get_analysis', 'get_asset']
CEN-05 · tools_aceitaveis: ['search_knowledge', 'list_analyses', 'get_baseline', 'get_current_user']
CEN-05 · exige knowledge?: False

precedência: a hipótese elétrica se confirma ou cai na banda de 2x f-linha
precedência: 2x f-linha = 120 Hz vem de line_frequency_hz=60, não de conhecimento geral


**O que o dado mostra.** `kb_guid_003` diz, literalmente, que falhas elétricas geram componentes em
*2x a frequência de linha (120 Hz em 60 Hz)* e que **espectros parciais nessa banda tornam a
inferência incerta** — que é, palavra por palavra, a conclusão que o gabarito de CEN-05 cobra. O
cenário lista `search_knowledge` em `tools_aceitaveis` (logo, não penaliza quem busca), mas não
declara nenhuma exigência de `knowledge` nem o documento como evidência.

Isso **não é uma contradição**: a precedência de CEN-05 exige que os 120 Hz venham de
`line_frequency_hz=60` do ativo, *"não de conhecimento geral"*, e um documento da base **não é**
conhecimento geral — é fonte citável. As duas rotas são válidas e o gabarito atual aceita as duas
sem medir nenhuma. O que se perde é a chance de exercitar `kb_guid_003` como fonte, que é a
proposta **L5** do `.md` — e a alternativa barata (declarar o documento em `CEN-05`) é curadoria,
não minha.

**A contagem errada de `CENARIOS §5.3` vazou para dentro do corpus.** A tabela acima mostra os
quatro primeiros documentos citados também por `aut_03`: é o campo
`estado_esperado.docs_existentes`, que enumera **quatro**. Não afeta o gabarito de AUT-03 (a busca
por `reprocesso` volta vazia em qualquer caso), mas é um YAML afirmando um fato do mundo que o
mundo não confirma. Levantado aqui; corrigir é curadoria.


---
## 6. A figura

Heatmap 7 × 5 com a contagem de cenários por célula. Quatro classes visuais, e a distinção que a
figura existe para carregar é **impossível ≠ vazia-mas-possível**: uma célula cinza não é dívida,
uma célula com borda é.


In [13]:
# Paleta do projeto (mesma do nb01). Azul sequencial para cobertura; âmbar = lacuna real;
# verde = coberta por outro caminho; cinza = impossível. Nunca só cor: todo estado tem rótulo.
TINTA, TINTA2, SUPERFICIE = "#0b0b0b", "#52514e", "#fcfcfb"
AZUL = ["#dbe8f8", "#a9c8ee", "#6fa3e2", "#2a78d6"]
AMBAR, AMBAR_F = "#eda100", "#fdf3d6"
VERDE, VERDE_F = "#1baf7a", "#e3f4ed"
CINZA, CINZA_F = "#c9c8c4", "#eeedea"


def tom(n: int) -> str:
    return AZUL[0 if n <= 2 else 1 if n <= 5 else 2 if n <= 10 else 3]


formas, notas = [], []
for iy, linha in enumerate(LINHAS):
    for ix, modo in enumerate(MODOS):
        ids = sorted(CURTO[i] for i in celulas[(linha, modo)])
        if ids:
            fundo, borda = tom(len(ids)), SUPERFICIE
            cor_texto = TINTA if len(ids) <= 10 else SUPERFICIE
            topo = f"<b>{len(ids)}</b>"
            rodape = " · ".join(ids) if len(ids) <= 3 else "cenários"
        elif not POSSIVEL[(linha, modo)]:
            fundo, borda, cor_texto = CINZA_F, CINZA_F, TINTA2
            topo, rodape = "—", "impossível"
        else:
            balde = CLASSIFICACAO[(linha, modo)][0]
            lacuna = balde == "lacuna real"
            fundo, borda = (AMBAR_F, AMBAR) if lacuna else (VERDE_F, VERDE)
            cor_texto = TINTA
            topo = "<b>0</b>"
            rodape = CLASSIFICACAO[(linha, modo)][1] if lacuna else "≡ " + CLASSIFICACAO[(linha, modo)][1]

        formas.append(dict(type="rect", x0=ix - 0.47, x1=ix + 0.47, y0=iy - 0.42, y1=iy + 0.42,
                           fillcolor=fundo, line=dict(color=borda, width=2), layer="below"))
        notas.append(dict(x=ix, y=iy - 0.11, text=topo, showarrow=False,
                          font=dict(size=17, color=cor_texto, family="Inter, Helvetica, sans-serif")))
        notas.append(dict(x=ix, y=iy + 0.19, text=rodape, showarrow=False,
                          font=dict(size=9, color=cor_texto, family="Inter, Helvetica, sans-serif")))

fig = go.Figure()
for nome, cor, fundo in [("coberta (nº de cenários)", AZUL[3], AZUL[3]),
                         ("lacuna real", AMBAR, AMBAR_F),
                         ("coberta por outro caminho", VERDE, VERDE_F),
                         ("impossível", CINZA_F, CINZA_F)]:
    fig.add_scatter(x=[None], y=[None], mode="markers", name=nome,
                    marker=dict(size=13, color=fundo, symbol="square",
                                line=dict(color=cor, width=2)))

fig.update_layout(
    shapes=formas, annotations=notas,
    title=dict(
        text="<b>Cobertura do corpus: categorias do TAPI §5 × StatusRetorno</b><br>"
             "<span style='font-size:12px;color:#52514e'>24 cenários · 89 exigências de "
             f"<i>ambiente.modos_exigidos</i> · {n_cob} de {n_poss} células possíveis cobertas "
             f"({n_cob / n_poss:.0%})</span>",
        x=0, xanchor="left", font=dict(size=17, color=TINTA)),
    xaxis=dict(tickmode="array", tickvals=list(range(5)),
               ticktext=[ROTULO_MODO[m] for m in MODOS], side="top", range=[-0.6, 4.6],
               showgrid=False, zeroline=False, tickfont=dict(color=TINTA, size=12)),
    yaxis=dict(tickmode="array", tickvals=list(range(7)), ticktext=LINHAS,
               autorange="reversed", range=[-0.6, 6.6], showgrid=False, zeroline=False,
               tickfont=dict(color=TINTA, size=12)),
    legend=dict(orientation="h", y=-0.09, x=0, font=dict(color=TINTA2, size=11), title=None),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
    margin=dict(l=130, r=28, t=112, b=64), height=560, width=980,
    font=dict(family="Inter, Helvetica, sans-serif"),
)

destino = RAIZ / "figures" / "fig02_matriz_cobertura.png"
fig.write_image(destino, scale=2)
print("gravado:", destino.relative_to(RAIZ))
fig.show()


gravado: figures/fig02_matriz_cobertura.png


---
## 7. Conclusão em números — onde o corpus está concentrado

A pergunta que a auditoria tem de responder sem adjetivo: **quanto do corpus roda com o dado
íntegro?** A distribuição nominal da API (`data/seed.json`) é `60/15/10/8/7`. Se o corpus fosse
uma amostra do ambiente, os modos exigidos seguiriam isso de perto.


In [14]:
NOMINAL = json.loads((RAIZ / "inteli-tractian-project" / "data" / "seed.json").read_text())["distribution"]
por_modo = Counter(m for _, r in EXIG.iterrows() for m in r["modos"])
total = sum(por_modo.values())

resumo = pd.DataFrame({
    "exigências": pd.Series(por_modo),
    "% do corpus": pd.Series({m: por_modo[m] / total for m in MODOS}),
    "nominal da API": pd.Series(NOMINAL),
}).reindex(MODOS)
resumo["desvio p.p."] = ((resumo["% do corpus"] - resumo["nominal da API"]) * 100).round(1)
display(resumo.style.format({"% do corpus": "{:.1%}", "nominal da API": "{:.0%}"}))

por_proc = EXIG.explode("modos").pivot_table(
    index="procedencia", columns="modos", values="cenário", aggfunc="count", fill_value=0
).reindex(columns=MODOS, fill_value=0)
por_proc["total"] = por_proc.sum(axis=1)
por_proc["% complete"] = (por_proc["complete"] / por_proc["total"] * 100).round(1)
display(por_proc)

sem_degradacao = [c["id"] for c in CENARIOS
                  if all(m == "complete" for e in c["ambiente"]["modos_exigidos"] for m in e["modos"])]
print(f"cenários que não exercitam nenhum modo degradado: {len(sem_degradacao)}/{len(CENARIOS)} "
      f"({len(sem_degradacao) / len(CENARIOS):.0%})")
print("  autorais:", sum(1 for i in sem_degradacao if i.startswith("aut")), "/ 8")
print("  oficiais:", sum(1 for i in sem_degradacao if i.startswith("cen")), "/ 16")
print("\ncategorias nunca exigidas por cenário nenhum:",
      sorted(set(LINHA_DA_CAT) - set(EXIG["categoria"])))
print("`get_company` em tools_esperadas:",
      sum(1 for c in CENARIOS if "get_company" in c["gabarito"].get("tools_esperadas", [])))


,exigências,% do corpus,nominal da API,desvio p.p.
complete,74,83.1%,60%,23.100000
partial,8,9.0%,15%,-6.000000
inconclusive,2,2.2%,10%,-7.800000
conflict,3,3.4%,8%,-4.600000
unavailable,2,2.2%,7%,-4.800000


modos,complete,partial,inconclusive,conflict,unavailable,total,% complete
procedencia,,,,,,,
autoral,27,0,0,0,0,27,100.0
oficial,47,8,2,3,2,62,75.8


cenários que não exercitam nenhum modo degradado: 15/24 (62%)
  autorais: 8 / 8
  oficiais: 7 / 16

categorias nunca exigidas por cenário nenhum: ['company']
`get_company` em tools_esperadas: 0


### A leitura, com os números

**Sim, o corpus é enviesado para `complete` — em 23 pontos percentuais.** 74 das 89 exigências
(83,1%) pedem `complete`, contra 60% da distribuição nominal da API. As outras quatro colunas
somam 15 exigências: `partial` 8, `conflict` 3, `inconclusive` 2, `unavailable` 2.

Três recortes que dizem mais que o agregado:

1. **Os 8 cenários autorais exigem `complete` em 27 de 27 exigências — 100%.** Eles não tocam o
   eixo `StatusRetorno`. Isso é coerente com o que se propuseram a cobrir (`CENARIOS §4.1`:
   negativo verdadeiro, 404, escopo entre empresas, premissa falsa, ambiguidade), mas significa
   que **toda a cobertura de degradação do corpus é carregada pelos 16 oficiais**, e mesmo neles
   `complete` é 47 de 62 (75,8%).
2. **15 dos 24 cenários (62,5%) rodam sem nenhum modo degradado.** A degradação vive em nove
   cenários, e metade das ocorrências vem de dois — CEN-01 e CEN-10, que são o mesmo ativo
   (`asset_G501`) com o mesmo bloco de overrides. Tirando esses dois, sobram sete cenários
   segurando quatro colunas.
3. **A categoria `company` não é exigida por cenário nenhum, e `get_company` não aparece em
   nenhuma `tools_esperadas`.** A linha `Contexto` está coberta inteiramente por `assets` (a
   listagem por empresa, em AUT-04/05/07). O endpoint `GET /companies/{id}` é o único de leitura
   do contrato que o corpus nunca toca.

**O que isso não quer dizer.** Um corpus com 60% de `complete` seria uma *amostra do ambiente*, e
não é isso que um corpus de avaliação deve ser: 19 dos 24 cenários são dado-dependentes, e num
cenário dado-dependente o modo degradado **destrói o cenário** em vez de enriquecê-lo — AUT-01 sob
`analyses=inconclusive` deixa de ser um negativo verdadeiro (`CENARIOS §8.3`). O viés é em parte
uma consequência necessária do desenho.

**O que ele quer dizer.** A parcela de degradação que *não* destrói cenário está subutilizada, e
as quatro lacunas reais da §4 estão todas aí: `model` em `partial` (gabarito já escrito em dois
ramos, faltando só uma seed), o cadastro do ativo sumindo, o inconclusivo por linha ausente, e o
conflito irresolúvel em fonte de linha única. E a coluna `INDISPONÍVEL` inteira depende de **dois
cenários sobre um único ativo** — se `asset_G501` saísse do corpus, o corpus perderia a coluna.


---
## 8. Saídas e acoplamento declarado

| Saída | Onde |
|---|---|
| matriz preenchida com ids | `docs/lacunas_justificadas.md §2` (tabela da §2 acima) |
| classificação das 12 células vazias | `docs/lacunas_justificadas.md §4` (tabela da §4 acima) |
| propostas L1–L5 em prosa | `docs/lacunas_justificadas.md §5` — escritas à mão, não saem daqui |
| figura | `figures/fig02_matriz_cobertura.png` (§6 acima) |

**O notebook não escreve o `.md`.** As duas tabelas em markdown foram geradas aqui e **copiadas**
para `docs/lacunas_justificadas.md`, como o `nb01` fez com `docs/catalogo_respostas.md`. Quem
reexecutar e vir número diferente precisa atualizar o documento à mão; hoje eles conferem.

**O notebook não edita `scenarios/*.yaml`.** As lacunas reais viram **proposta escrita**, não
arquivo: criar cenário é decisão de curadoria do dono do projeto, e a razão está em
`scenarios/README.md` — recriar até tudo passar transforma o corpus numa descrição do que esta API
deixa fácil.

**Toda figura do projeto vem de notebook versionado.** Nenhum print de tela.
